# Imports

In [ ]:
!pip install -q opencv-python-headless

import os, sys, json, time, random, math
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import torchvision.transforms as T

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Configurations

In [ ]:
DATA_DIR   = '/kaggle/input/datasets/mennaezzelarab/shop-dataset/Shop DataSet'  

# Folder names inside DATA_DIR that map to labels
CLASS_TO_IDX = {'non shop lifters': 0, 'shop lifters': 1}  # adjust if your folders differ

# ── Preprocessing ──────────────────────────────────────────────────────────
N_FRAMES    = 32      # fixed frame count per video (try 8, 16 or 32)
IMG_SIZE    = 112     # spatial resize H=W (112 is fast; 224 is more accurate)

# ── Training ───────────────────────────────────────────────────────────────
MODEL_NAME  = '3dcnn'   # '3dcnn' | 'cnn_rnn' | 'transformer'
BATCH_SIZE  = 8
LR         = 3e-4   # was 1e-3 — too high, caused the loss spikes you saw
MAX_EPOCHS = 50
PATIENCE   = 10
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.15
NUM_WORKERS = 2         # Kaggle allows 2 workers safely
SEED        = 42

# POS_WEIGHT: up-weights the stealing class in the loss.
# Rule of thumb: set to (# not_stealing / # stealing).
# Leave as None to auto-compute after EDA.
POS_WEIGHT  = 1.64

CKPT_PATH = f'/kaggle/working/best_{MODEL_NAME}.pt'

SUPPORTED_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv'}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Config loaded.')

# Data Exploration

In [ ]:
def gather_video_paths(data_dir):
    """Return {class_name: [Path, ...]} from data_dir/class_name/ layout."""
    root = Path(data_dir)
    class_paths = defaultdict(list)
    for cls_dir in sorted(root.iterdir()):
        if not cls_dir.is_dir():
            continue
        for f in cls_dir.rglob('*'):
            if f.suffix.lower() in SUPPORTED_EXTS:
                class_paths[cls_dir.name].append(f)
    return dict(class_paths)


def probe_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    info = dict(
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        fps         = cap.get(cv2.CAP_PROP_FPS),
    )
    cap.release()
    return info


def sample_frames_uniform(path, n=6):
    cap   = cv2.VideoCapture(str(path))
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    idxs  = np.linspace(0, total - 1, n, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


# ── Run EDA ────────────────────────────────────────────────────────────────
class_paths = gather_video_paths(DATA_DIR)
assert class_paths, f'No videos found in {DATA_DIR}. Check DATA_DIR and folder names.'

all_infos = {}
print('=' * 60)
print('  VIDEO DATASET — EDA')
print('=' * 60)

for cls_name, paths in class_paths.items():
    infos = [probe_video(p) for p in paths]
    infos = [i for i in infos if i is not None]
    all_infos[cls_name] = infos
    fc = [i['frame_count'] for i in infos]
    ws = [i['width']       for i in infos]
    hs = [i['height']      for i in infos]
    print(f'\n  Class: {cls_name!r}  ({len(paths)} videos)')
    print(f'  Frames — min:{min(fc)}  max:{max(fc)}  mean:{np.mean(fc):.1f}  std:{np.std(fc):.1f}')
    print(f'  Size   — W:{set(ws)}  H:{set(hs)}')

# ── Charts ─────────────────────────────────────────────────────────────────
classes = list(all_infos.keys())
counts  = [len(v) for v in all_infos.values()]
colors  = ['#E8593C', '#3B8BD4', '#1D9E75', '#BA7517']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset Overview', fontsize=13, fontweight='bold')

# Class balance
axes[0].bar(classes, counts, color=colors[:len(classes)])
axes[0].set_title('Class balance')
axes[0].set_ylabel('# videos')
for i, c in enumerate(counts):
    axes[0].text(i, c + 0.2, str(c), ha='center', fontweight='bold')

# Frame count histogram
for idx, (cls_name, infos) in enumerate(all_infos.items()):
    fc = [i['frame_count'] for i in infos]
    axes[1].hist(fc, bins=20, alpha=0.65, label=cls_name, color=colors[idx])
axes[1].set_title('Frame count distribution')
axes[1].set_xlabel('Frames / video')
axes[1].legend()

# Resolution scatter
for idx, (cls_name, infos) in enumerate(all_infos.items()):
    axes[2].scatter([i['width'] for i in infos], [i['height'] for i in infos],
                    label=cls_name, alpha=0.6, color=colors[idx])
axes[2].set_title('Resolution W × H')
axes[2].set_xlabel('Width')
axes[2].set_ylabel('Height')
axes[2].legend()

plt.tight_layout()
plt.show()

# ── Sample frames ──────────────────────────────────────────────────────────
for cls_name, paths in class_paths.items():
    frames = sample_frames_uniform(paths[0], n=6)
    if not frames:
        continue
    fig2, axs = plt.subplots(1, len(frames), figsize=(3 * len(frames), 3))
    fig2.suptitle(f'Sample frames — {cls_name!r}: {paths[0].name}', fontsize=10)
    for ax, fr in zip(axs, frames):
        ax.imshow(fr); ax.axis('off')
    plt.tight_layout(); plt.show()

# ── Recommendations ────────────────────────────────────────────────────────
all_fc        = [i['frame_count'] for infos in all_infos.values() for i in infos]
rec_frames    = int(np.clip(np.percentile(all_fc, 25), 8, 64))
imbalance     = max(counts) / max(min(counts), 1)
auto_pw       = round(max(counts) / max(min(counts), 1), 2)

print('\n' + '=' * 60)
print('  RECOMMENDATIONS')
print('=' * 60)
print(f'  Suggested N_FRAMES  : {rec_frames}')
print(f'  Suggested IMG_SIZE  : 112 (fast) or 224 (accurate)')
print(f'  Class imbalance     : {imbalance:.1f}x')
if imbalance > 1.5:
    print(f'  ⚠ Set POS_WEIGHT = {auto_pw} in Config cell to compensate')
    POS_WEIGHT = auto_pw
    print(f'    (auto-set POS_WEIGHT = {POS_WEIGHT})')
else:
    print('  ✓ Classes are balanced — POS_WEIGHT not needed')

In [ ]:
print(f'POS_WEIGHT = {POS_WEIGHT}')  

In [ ]:
# Run this after EDA to check real counts
for cls_name, infos in all_infos.items():
    print(f'{cls_name}: {len(infos)} videos')

# Then manually set POS_WEIGHT
# Formula: num_non_shoplifters / num_shoplifters
# e.g. if 500 non-shoplifters and 200 shoplifters → POS_WEIGHT = 500/200 = 2.5
POS_WEIGHT = None  # will be set below
counts_dict = {cls: len(infos) for cls, infos in all_infos.items()}
POS_WEIGHT = round(counts_dict['non shop lifters'] / counts_dict['shop lifters'], 2)
print(f'POS_WEIGHT set to: {POS_WEIGHT}')

In [ ]:
print(f'POS_WEIGHT = {POS_WEIGHT}') 

# Dataset Class (sampling + augmentation)

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ── sampling ───────────────────────────────────────────────────────────────

def _read_frames_uniform(cap, n):
    """Sample exactly n frames uniformly from an open VideoCapture."""
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    idxs  = np.linspace(0, total - 1, n, dtype=int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            frames.append(frames[-1] if frames
                          else np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    return frames


def _read_frames_random_window(cap, n):
    """Random contiguous window of n frames (training augmentation)."""
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= n:
        return _read_frames_uniform(cap, n)
    start = random.randint(0, total - n)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(n):
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            frames.append(frames[-1] if frames
                          else np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    return frames


# ── augmentation ───────────────────────────────────────────────────────────

def augment_clip(frames, target_size):
    do_flip      = random.random() < 0.5
    do_jitter    = random.random() < 0.7    # more frequent
    do_grayscale = random.random() < 0.2    # occasionally train in grayscale
    do_reverse   = random.random() < 0.3    # reverse temporal order
    scale        = random.uniform(0.75, 1.0)
    jitter       = T.ColorJitter(brightness=0.4, contrast=0.4,
                                 saturation=0.4, hue=0.1)

    # decide crop position once for the whole clip
    if scale < 1.0:
        sample_h, sample_w = frames[0].shape[:2]
        nh = int(sample_h * scale)
        nw = int(sample_w * scale)
        top  = random.randint(0, sample_h - nh)
        left = random.randint(0, sample_w - nw)
    else:
        top = left = nh = nw = 0

    if do_reverse:
        frames = frames[::-1]

    out = []
    for frame in frames:
        if scale < 1.0:
            frame = frame[top:top+nh, left:left+nw]
        frame = cv2.resize(frame, (target_size, target_size))
        if do_flip:
            frame = frame[:, ::-1, :].copy()
        if do_jitter:
            frame = np.array(jitter(T.ToPILImage()(frame)))
        if do_grayscale:
            gray  = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            frame = np.stack([gray, gray, gray], axis=-1)
        out.append(frame)
    return out


def normalize_clip(frames, target_size):
    """Resize, convert to float32, apply ImageNet normalization."""
    out = []
    for frame in frames:
        frame = cv2.resize(frame, (target_size, target_size))
        frame = frame.astype(np.float32) / 255.0
        frame = (frame - IMAGENET_MEAN) / IMAGENET_STD
        out.append(frame)
    return out


# ── Dataset ────────────────────────────────────────────────────────────────

class VideoDataset(Dataset):
    """
    Returns (frames_tensor, label) where
      frames_tensor : (T, C, H, W)  float32
      label         : int  0 or 1
    """
    def __init__(self, data_dir, n_frames=N_FRAMES, target_size=IMG_SIZE,
                 augment=False, random_window=False):
        self.n_frames     = n_frames
        self.target_size  = target_size
        self.augment      = augment
        self.random_window = random_window
        self.samples      = []          # [(Path, label), ...]
        self._load(Path(data_dir))

    def _load(self, root):
        for cls_dir in sorted(root.iterdir()):
            if not cls_dir.is_dir():
                continue
            label = CLASS_TO_IDX.get(cls_dir.name, -1)
            if label == -1:
                print(f'  [WARN] Unknown folder: {cls_dir.name!r} — skipping')
                continue
            for f in cls_dir.rglob('*'):
                if f.suffix.lower() in SUPPORTED_EXTS:
                    self.samples.append((f, label))
        print(f'  VideoDataset: {len(self.samples)} videos')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        cap = cv2.VideoCapture(str(path))
        frames = (_read_frames_random_window(cap, self.n_frames)
                  if self.random_window else
                  _read_frames_uniform(cap, self.n_frames))
        cap.release()

        if self.augment:
            frames = augment_clip(frames, self.target_size)

        frames = normalize_clip(frames, self.target_size)

        # (T, H, W, C) → (T, C, H, W)
        arr    = np.stack(frames, axis=0).astype(np.float32)
        tensor = torch.from_numpy(arr).permute(0, 3, 1, 2)
        return tensor, label

    def class_weights(self):
        labels  = [lbl for _, lbl in self.samples]
        counts  = torch.bincount(torch.tensor(labels)).float()
        weights = 1.0 / counts
        return weights[torch.tensor(labels)]


print('Dataset class defined ✓')

# Build DataLoaders

In [ ]:
def make_dataloaders(data_dir, augment_train=True, balance=True):
    """
    Stratified train / val / test split.
    Returns dict with 'train', 'val', 'test' DataLoaders.
    """
    # base dataset (no augmentation) — used for splitting + val/test
    base_ds = VideoDataset(data_dir, augment=False, random_window=False)

    # stratified split
    class_indices = defaultdict(list)
    for i, (_, lbl) in enumerate(base_ds.samples):
        class_indices[lbl].append(i)

    train_idx, val_idx, test_idx = [], [], []
    for lbl, idxs in class_indices.items():
        random.shuffle(idxs)
        n      = len(idxs)
        n_test = math.ceil(n * TEST_SPLIT)
        n_val  = math.ceil(n * VAL_SPLIT)
        test_idx  += idxs[:n_test]
        val_idx   += idxs[n_test:n_test + n_val]
        train_idx += idxs[n_test + n_val:]

    # training dataset WITH augmentation
    train_ds = VideoDataset(data_dir, augment=augment_train, random_window=True)

    train_sub = Subset(train_ds, train_idx)
    val_sub   = Subset(base_ds,  val_idx)
    test_sub  = Subset(base_ds,  test_idx)

    sampler = None
    if balance:
        w       = base_ds.class_weights()
        sampler = WeightedRandomSampler(w[train_idx], len(train_idx), replacement=True)

    loaders = {
        'train': DataLoader(train_sub, batch_size=BATCH_SIZE,
                            sampler=sampler, shuffle=(sampler is None),
                            num_workers=NUM_WORKERS, pin_memory=True),
        'val':   DataLoader(val_sub,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
        'test':  DataLoader(test_sub,  batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    }
    print(f'  Splits — train:{len(train_sub)}  val:{len(val_sub)}  test:{len(test_sub)}')
    return loaders, base_ds, test_idx


loaders, base_ds, test_idx = make_dataloaders(DATA_DIR)

# Quick shape check
sample_x, sample_y = next(iter(loaders['train']))
print(f'  Batch shape: {sample_x.shape}  labels: {sample_y.tolist()}')
# Expected: torch.Size([BATCH_SIZE, N_FRAMES, 3, IMG_SIZE, IMG_SIZE])

# Save test paths for later use in prediction cell
test_paths = [base_ds.samples[i] for i in test_idx]
print(f'  Test set: {len(test_paths)} videos')
for path, label in test_paths[:5]:
    cls = [k for k, v in CLASS_TO_IDX.items() if v == label][0]
    print(f'    [{cls}] {path.name}')

# Model Architectures

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MODEL 1 — 3D CNN
# Input: (B, T, C, H, W)  →  permuted to (B, C, T, H, W) for Conv3d
# ═══════════════════════════════════════════════════════════════════

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=(3,3,3), stride=(1,1,1), padding=(1,1,1)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ThreeDCNN(nn.Module):
    def __init__(self, base_ch=64, dropout=0.3, **_):
        super().__init__()
        self.encoder = nn.Sequential(
            # Block 1 — gentle start, no temporal downsampling yet
            ConvBlock3D(3,         base_ch,   (3,3,3), (1,1,1), (1,1,1)),
            nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2)),   # spatial only

            # Block 2
            ConvBlock3D(base_ch,   base_ch*2, (3,3,3), (1,1,1), (1,1,1)),
            nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2)),   # spatial + temporal

            # Block 3
            ConvBlock3D(base_ch*2, base_ch*4, (3,3,3), (1,1,1), (1,1,1)),
            nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2)),

            # Block 4
            ConvBlock3D(base_ch*4, base_ch*4, (3,3,3), (1,1,1), (1,1,1)),
            nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2)),
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(base_ch * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1, 3, 4)   # (B,T,C,H,W) → (B,C,T,H,W)
        return self.head(self.pool(self.encoder(x)))

# ═══════════════════════════════════════════════════════════════════
# MODEL 2 — CNN + RNN
# Per-frame 2D CNN encoder → LSTM over the frame sequence
# ═══════════════════════════════════════════════════════════════════

class FrameCNN(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True),
            nn.Conv2d(32,  64,  3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64,  128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256, out_dim), nn.ReLU(True),
        )
    def forward(self, x):
        return self.net(x)

class CNNWithRNN(nn.Module):
    def __init__(self, feat_dim=256, hidden=512, n_layers=2, dropout=0.5, **_):
        super().__init__()
        self.cnn  = FrameCNN(out_dim=feat_dim)
        self.lstm = nn.LSTM(feat_dim, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 128), nn.ReLU(True),
            nn.Linear(128, 1),
        )
    def forward(self, x):                       # (B, T, C, H, W)
        B, T, C, H, W = x.shape
        feats = self.cnn(x.view(B*T, C, H, W)).view(B, T, -1)   # (B, T, feat_dim)
        out, _ = self.lstm(feats)               # (B, T, hidden)
        return self.head(out[:, -1])            # last step → (B, 1)


# ═══════════════════════════════════════════════════════════════════
# MODEL 3 — Video Transformer  (ViViT-inspired, from scratch)
# Patch embedding per frame → Transformer blocks → CLS token
# ═══════════════════════════════════════════════════════════════════

class PatchEmbed(nn.Module):
    def __init__(self, img_size=112, patch_size=16, in_ch=3, d_model=192):
        super().__init__()
        assert img_size % patch_size == 0
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):           # (B*T, C, H, W)
        return self.proj(x).flatten(2).transpose(1, 2)   # (B*T, n_patches, d)

class TxBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        dim = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, d_model), nn.Dropout(dropout),
        )
    def forward(self, x):
        y, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + y
        return x + self.mlp(self.norm2(x))

class VideoTransformer(nn.Module):
    def __init__(self, n_frames=N_FRAMES, img_size=IMG_SIZE, patch_size=16,
                 d_model=192, n_heads=3, n_layers=4, dropout=0.1, **_):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, d_model)
        seq_len = n_frames * self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len + 1, d_model))
        self.blocks = nn.Sequential(*[TxBlock(d_model, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, 1))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):               # (B, T, C, H, W)
        B, T, C, H, W = x.shape
        p = self.patch_embed(x.view(B*T, C, H, W)).view(B, T * self.patch_embed.n_patches, -1)
        x = torch.cat([self.cls_token.expand(B, -1, -1), p], dim=1)
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.norm(self.blocks(x))
        return self.head(x[:, 0])       # CLS token → (B, 1)


# ── Factory ────────────────────────────────────────────────────────────────

def build_model(name):
    cfg = dict(n_frames=N_FRAMES, img_size=IMG_SIZE)
    models = {'3dcnn': ThreeDCNN, 'cnn_rnn': CNNWithRNN, 'transformer': VideoTransformer}
    assert name in models, f'Unknown model: {name}. Choose from {list(models)}'
    m = models[name](**cfg)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  Model: {name}  |  Params: {n:,}')
    return m


print('All model classes defined ✓')
print('Available: 3dcnn · cnn_rnn · transformer')

# Training and Metrics

In [ ]:
def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tp = int(((preds==1) & (labels==1)).sum())
    tn = int(((preds==0) & (labels==0)).sum())
    fp = int(((preds==1) & (labels==0)).sum())
    fn = int(((preds==0) & (labels==1)).sum())
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    acc  = (tp + tn) / max(len(labels), 1)
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, tp=tp, tn=tn, fp=fp, fn=fn)


def run_epoch(model, loader, criterion, optimizer=None):
    """One training or evaluation pass. optimizer=None → eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, all_probs, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.float().to(DEVICE, non_blocking=True)

            logits = model(x).squeeze(1)          # (B,)
            loss   = criterion(logits, y)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(y)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(y.cpu().numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    m      = compute_metrics(probs, labels)
    m['loss'] = total_loss / max(len(labels), 1)
    return m


def train(model, loaders, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    pw        = torch.tensor([POS_WEIGHT]).to(DEVICE) if POS_WEIGHT else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

    # CosineAnnealing is much more stable than ReduceLROnPlateau for from-scratch models
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=max_epochs, eta_min=1e-6)

    best_f1, patience_ctr = 0.0, 0
    history = []

    print(f'\n  {"Ep":>3}  {"TrLoss":>7}  {"TrF1":>6}  {"VlLoss":>7}  {"VlF1":>6}  {"VlAcc":>6}  {"LR":>8}  {"GradNorm":>9}')
    print('  ' + '-'*72)

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        # ── training pass with grad norm tracking ──────────────────
        model.train()
        total_loss, all_probs, all_labels = 0.0, [], []
        total_grad_norm = 0.0
        n_batches = 0

        for x, y in loaders['train']:
            x = x.to(DEVICE, non_blocking=True)
            y = y.float().to(DEVICE, non_blocking=True)

            logits = model(x).squeeze(1)
            loss   = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()

            # track grad norm BEFORE clipping to detect instability
            grad_norm = sum(p.grad.norm().item()**2
                            for p in model.parameters()
                            if p.grad is not None) ** 0.5
            total_grad_norm += grad_norm
            n_batches += 1

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(y)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(y.cpu().numpy())

        scheduler.step()

        probs  = np.concatenate(all_probs)
        labels = np.concatenate(all_labels)
        tr     = compute_metrics(probs, labels)
        tr['loss'] = total_loss / max(len(labels), 1)
        avg_grad_norm = total_grad_norm / max(n_batches, 1)

        val = run_epoch(model, loaders['val'], criterion)
        history.append({'epoch': epoch, 'train': tr, 'val': val,
                        'lr': scheduler.get_last_lr()[0],
                        'grad_norm': avg_grad_norm})

        lr_now = scheduler.get_last_lr()[0]
        print(f"  {epoch:3d}  {tr['loss']:7.4f}  {tr['f1']:6.4f}  "
              f"{val['loss']:7.4f}  {val['f1']:6.4f}  {val['acc']:6.4f}  "
              f"{lr_now:8.2e}  {avg_grad_norm:9.4f}  ({time.time()-t0:.1f}s)")

        # warn if model is outputting near-constant predictions
        pred_std = np.std(probs)
        if pred_std < 0.02:
            print(f'  ⚠ WARNING: prediction std={pred_std:.4f} — model may be collapsing')

        if val['f1'] > best_f1:
            best_f1, patience_ctr = val['f1'], 0
            torch.save({'epoch': epoch, 'state': model.state_dict(),
                        'metrics': val}, CKPT_PATH)
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping (best val F1 = {best_f1:.4f})')
                break

    return history


def plot_history(history):
    epochs = [h['epoch'] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split, color in [('train','#E8593C'), ('val','#3B8BD4')]:
        axes[0].plot(epochs, [h[split]['loss'] for h in history],
                     label=split, color=color)
        axes[1].plot(epochs, [h[split]['f1']   for h in history],
                     label=split, color=color)
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].set_title('F1 score'); axes[1].legend()
    plt.tight_layout(); plt.show()


print('Trainer functions defined ✓')

# Train

**3D CNN**

In [ ]:
# Build model
model = build_model(MODEL_NAME).to(DEVICE)

# Train
history = train(model, loaders)

# Plot training curves
plot_history(history)

**RNN + CNN**

In [ ]:
MODEL_NAME = 'cnn_rnn'
CKPT_PATH  = f'/kaggle/working/best_{MODEL_NAME}.pt' 
print(f'CKPT_PATH → {CKPT_PATH}')

In [ ]:
# Build model
model = build_model(MODEL_NAME).to(DEVICE)

# Train
history = train(model, loaders)

# Plot training curves
plot_history(history)

**Transformer**

In [ ]:
MODEL_NAME  = 'transformer'   # '3dcnn' | 'cnn_rnn' | 'transformer'
CKPT_PATH  = f'/kaggle/working/best_{MODEL_NAME}.pt' 
print(f'CKPT_PATH → {CKPT_PATH}')

In [ ]:
# Build model
model = build_model(MODEL_NAME).to(DEVICE)

# Train
history = train(model, loaders)

# Plot training curves
plot_history(history)

# Model Comparison

In [ ]:
# ── Compare all three models ───────────────────────────────────────
comparison = []

for name in ['3dcnn', 'cnn_rnn', 'transformer']:
    ckpt_path = f'/kaggle/working/best_{name}.pt'
    if not Path(ckpt_path).exists():
        print(f'  [{name}] no checkpoint found — skipping')
        continue

    # rebuild and load
    m = build_model(name).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    m.load_state_dict(ckpt['state'])

    # evaluate on test set
    pw        = torch.tensor([POS_WEIGHT]).to(DEVICE) if POS_WEIGHT else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    test_m    = run_epoch(m, loaders['test'], criterion)

    comparison.append(dict(
        model     = name,
        epoch     = ckpt['epoch'],
        val_f1    = round(ckpt['metrics']['f1'],  4),
        test_f1   = round(test_m['f1'],   4),
        test_acc  = round(test_m['acc'],  4),
        test_prec = round(test_m['prec'], 4),
        test_rec  = round(test_m['rec'],  4),
    ))
    print(f'  [{name}] loaded from epoch {ckpt["epoch"]}')

# print table
print(f'\n  {"Model":<15}  {"Best Epoch":>10}  {"Val F1":>7}  '
      f'{"Test F1":>8}  {"Accuracy":>9}  {"Precision":>10}  {"Recall":>7}')
print('  ' + '-'*72)
for r in comparison:
    print(f'  {r["model"]:<15}  {r["epoch"]:>10}  {r["val_f1"]:>7.4f}  '
          f'{r["test_f1"]:>8.4f}  {r["test_acc"]:>9.4f}  '
          f'{r["test_prec"]:>10.4f}  {r["test_rec"]:>7.4f}')

# bar chart
if comparison:
    names     = [r['model']    for r in comparison]
    test_f1s  = [r['test_f1']  for r in comparison]
    test_accs = [r['test_acc'] for r in comparison]

    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.2, test_f1s,  0.35, label='Test F1',       color='#3B8BD4')
    ax.bar(x + 0.2, test_accs, 0.35, label='Test Accuracy', color='#1D9E75')
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1)
    ax.set_title('Model comparison — test set')
    ax.legend()
    for i, (f1, acc) in enumerate(zip(test_f1s, test_accs)):
        ax.text(i - 0.2, f1  + 0.01, f'{f1:.3f}',  ha='center', fontsize=9)
        ax.text(i + 0.2, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

# Evaluate on Test Set

In [ ]:
# Explicitly choose which model to evaluate
EVAL_MODEL_NAME = '3dcnn'   # change to whichever you want to evaluate
EVAL_CKPT = f'/kaggle/working/best_{EVAL_MODEL_NAME}.pt'

m = build_model(EVAL_MODEL_NAME).to(DEVICE)
ckpt = torch.load(EVAL_CKPT, map_location=DEVICE)
m.load_state_dict(ckpt['state'])
print(f'  Loaded {EVAL_MODEL_NAME} from epoch {ckpt["epoch"]} (val F1={ckpt["metrics"]["f1"]:.4f})')

pw        = torch.tensor([POS_WEIGHT]).to(DEVICE) if POS_WEIGHT else None
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
test_m    = run_epoch(m, loaders['test'], criterion)

print(f'\n  ── TEST RESULTS — {EVAL_MODEL_NAME} ───────────────')
print(f'  Accuracy  : {test_m["acc"]:.4f}')
print(f'  Precision : {test_m["prec"]:.4f}')
print(f'  Recall    : {test_m["rec"]:.4f}')
print(f'  F1 score  : {test_m["f1"]:.4f}')

tp, tn, fp, fn = test_m['tp'], test_m['tn'], test_m['fp'], test_m['fn']
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: normal', 'Pred: stealing'])
ax.set_yticklabels(['True: normal', 'True: stealing'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
ax.set_title(f'Confusion Matrix — {EVAL_MODEL_NAME}')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
# Explicitly choose which model to evaluate
EVAL_MODEL_NAME = 'cnn_rnn'   # change to whichever you want to evaluate
EVAL_CKPT = f'/kaggle/working/best_{EVAL_MODEL_NAME}.pt'

m = build_model(EVAL_MODEL_NAME).to(DEVICE)
ckpt = torch.load(EVAL_CKPT, map_location=DEVICE)
m.load_state_dict(ckpt['state'])
print(f'  Loaded {EVAL_MODEL_NAME} from epoch {ckpt["epoch"]} (val F1={ckpt["metrics"]["f1"]:.4f})')

pw        = torch.tensor([POS_WEIGHT]).to(DEVICE) if POS_WEIGHT else None
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
test_m    = run_epoch(m, loaders['test'], criterion)

print(f'\n  ── TEST RESULTS — {EVAL_MODEL_NAME} ───────────────')
print(f'  Accuracy  : {test_m["acc"]:.4f}')
print(f'  Precision : {test_m["prec"]:.4f}')
print(f'  Recall    : {test_m["rec"]:.4f}')
print(f'  F1 score  : {test_m["f1"]:.4f}')

tp, tn, fp, fn = test_m['tp'], test_m['tn'], test_m['fp'], test_m['fn']
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: normal', 'Pred: stealing'])
ax.set_yticklabels(['True: normal', 'True: stealing'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
ax.set_title(f'Confusion Matrix — {EVAL_MODEL_NAME}')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
# Explicitly choose which model to evaluate
EVAL_MODEL_NAME = 'transformer'   # change to whichever you want to evaluate
EVAL_CKPT = f'/kaggle/working/best_{EVAL_MODEL_NAME}.pt'

m = build_model(EVAL_MODEL_NAME).to(DEVICE)
ckpt = torch.load(EVAL_CKPT, map_location=DEVICE)
m.load_state_dict(ckpt['state'])
print(f'  Loaded {EVAL_MODEL_NAME} from epoch {ckpt["epoch"]} (val F1={ckpt["metrics"]["f1"]:.4f})')

pw        = torch.tensor([POS_WEIGHT]).to(DEVICE) if POS_WEIGHT else None
criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
test_m    = run_epoch(m, loaders['test'], criterion)

print(f'\n  ── TEST RESULTS — {EVAL_MODEL_NAME} ───────────────')
print(f'  Accuracy  : {test_m["acc"]:.4f}')
print(f'  Precision : {test_m["prec"]:.4f}')
print(f'  Recall    : {test_m["rec"]:.4f}')
print(f'  F1 score  : {test_m["f1"]:.4f}')

tp, tn, fp, fn = test_m['tp'], test_m['tn'], test_m['fp'], test_m['fn']
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred: normal', 'Pred: stealing'])
ax.set_yticklabels(['True: normal', 'True: stealing'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
ax.set_title(f'Confusion Matrix — {EVAL_MODEL_NAME}')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

In [ ]:
print(f'\n  Threshold sweep on VALIDATION set:')
print(f'  {"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"F1":>7}  {"Accuracy":>9}')
print('  ' + '-'*48)

# collect val probs
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in loaders['val']:
        logits = model(x.to(DEVICE)).squeeze(1)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y.numpy())

val_probs  = np.concatenate(all_probs)
val_labels = np.concatenate(all_labels)

best_thresh, best_f1 = 0.5, 0.0
for thresh in np.arange(0.3, 0.75, 0.05):
    m = compute_metrics(val_probs, val_labels, threshold=thresh)
    marker = ' ←' if m['f1'] > best_f1 else ''
    print(f'  {thresh:>10.2f}  {m["prec"]:>10.4f}  {m["rec"]:>8.4f}  '
          f'{m["f1"]:>7.4f}  {m["acc"]:>9.4f}{marker}')
    if m['f1'] > best_f1:
        best_f1   = m['f1']
        best_thresh = thresh

print(f'\n  Best threshold: {best_thresh:.2f}  (val F1 = {best_f1:.4f})')
BEST_THRESHOLD = best_thresh

# Predict a single video

In [ ]:
def predict_video(video_path, model, threshold=0.5):
    """Return (probability_of_stealing, label_string) for one video."""
    model.eval()
    cap    = cv2.VideoCapture(str(video_path))
    frames = _read_frames_uniform(cap, N_FRAMES)
    cap.release()

    frames = normalize_clip(frames, IMG_SIZE)
    arr    = np.stack(frames, axis=0).astype(np.float32)
    tensor = torch.from_numpy(arr).permute(0, 3, 1, 2).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).item()

    label = 'STEALING ⚠' if prob >= threshold else 'normal ✓'
    return prob, label


# ── Example usage ──────────────────────────────────────────────────────────
# Replace with any video path you want to test
TEST_VIDEO = '/kaggle/input/datasets/mennaezzelarab/shop-dataset/Shop DataSet/shop lifters/shop_lifter_120.mp4'   # <-- CHANGE

prob, label = predict_video(TEST_VIDEO, model)
print(f'  Video  : {Path(TEST_VIDEO).name}')
print(f'  P(stealing) = {prob:.4f}  →  {label}')

# Show the frames we used
cap    = cv2.VideoCapture(TEST_VIDEO)
frames = _read_frames_uniform(cap, 6)
cap.release()

fig, axs = plt.subplots(1, 6, figsize=(18, 3))
for ax, fr in zip(axs, frames):
    ax.imshow(fr); ax.axis('off')
fig.suptitle(f'Prediction: {label}  (p = {prob:.3f})', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Pick a video from the ACTUAL test set
# Change the index to inspect different test videos
TEST_IDX = 5   # <-- change this (0 to len(test_paths)-1)

test_video_path, true_label = test_paths[TEST_IDX]
true_class = [k for k, v in CLASS_TO_IDX.items() if v == true_label][0]

prob, pred_label = predict_video(test_video_path, model)

print(f'  Video       : {test_video_path.name}')
print(f'  True label  : {true_class}')
print(f'  P(stealing) : {prob:.4f}  →  {pred_label}')
correct = (prob >= 0.5) == bool(true_label)
print(f'  Result      : {"✓ correct" if correct else "✗ wrong"}')

# Show sampled frames
cap = cv2.VideoCapture(str(test_video_path))
frames = _read_frames_uniform(cap, 6)
cap.release()

fig, axs = plt.subplots(1, 6, figsize=(18, 3))
for ax, fr in zip(axs, frames):
    ax.imshow(fr); ax.axis('off')
fig.suptitle(f'True: {true_class}  |  Pred: {pred_label}  (p={prob:.3f})', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def predict_all_test_videos(model, test_paths, threshold=0.5):
    results = []
    model.eval()
    for path, true_label in test_paths:
        prob, pred_label = predict_video(path, model, threshold)
        correct = (prob >= threshold) == bool(true_label)
        true_class = [k for k, v in CLASS_TO_IDX.items() if v == true_label][0]
        results.append(dict(
            name=path.name,
            true=true_class,
            pred=pred_label,
            prob=round(prob, 4),
            correct=correct,
        ))
    return results


results = predict_all_test_videos(model, test_paths, threshold=BEST_THRESHOLD)

# ── Summary ────────────────────────────────────────────────────────
n_correct = sum(r['correct'] for r in results)
print(f'\n  Predicted {len(results)} test videos')
print(f'  Correct  : {n_correct} / {len(results)}  ({100*n_correct/len(results):.1f}%)')

# ── Per-video table ────────────────────────────────────────────────
print(f'\n  {"Video":<40}  {"True":<15}  {"Pred":<20}  {"P(steal)":>9}  {"OK?":>4}')
print('  ' + '-' * 95)
for r in results:
    tick = '✓' if r['correct'] else '✗'
    print(f'  {r["name"]:<40}  {r["true"]:<15}  {r["pred"]:<20}  {r["prob"]:>9.4f}  {tick:>4}')

# ── Visual grid (wrong predictions only) ──────────────────────────
wrong = [(test_paths[i], r) for i, r in enumerate(results) if not r['correct']]
if not wrong:
    print('\n  ✓ No wrong predictions!')
else:
    print(f'\n  Showing {len(wrong)} wrong predictions:')
    for (path, true_label), r in wrong:
        cap    = cv2.VideoCapture(str(path))
        frames = _read_frames_uniform(cap, 6)
        cap.release()

        fig, axs = plt.subplots(1, 6, figsize=(18, 3))
        for ax, fr in zip(axs, frames):
            ax.imshow(fr); ax.axis('off')
        fig.suptitle(
            f'✗ WRONG  |  True: {r["true"]}  →  Pred: {r["pred"]}  (p={r["prob"]:.3f})',
            fontsize=11, color='red'
        )
        plt.tight_layout()
        plt.show()